In [1]:
import geopandas as gpd

import pandas as pd

In [2]:
# config
pd.set_option('display.max_colwidth', None)

In [7]:
shapefile_path = "data/historic_airphoto_index/historic_airphoto_index.shp"

gdf = gpd.read_file(shapefile_path)

print(gdf.crs)

EPSG:3857


In [9]:
# convert to UTM 15
gdf = gdf.to_crs(epsg=26915)

print(gdf.crs)

EPSG:26915


In [10]:
#save a UTM copy of that shapefile just for kicks
gdf.to_file("data/historic_airphoto_index/historic_airphoto_index_utm.shp")

In [ ]:
#preview the headers and some of the data

gdf.columns

gdf

### adding columns to tracker table

columns to add: 

- download_filename
- output_filename

- geometry_utm (GeoSeries in EPSG:26915)
- ml_confidence
- gcp_count
- error_log

### filenames

In [12]:
# create a safe file name
gdf["download_filename"] = (
    gdf["filename"]
    .astype(str)
    .str.lstrip("/")
    .str.replace("/", "_")
)

In [13]:
# create a safe file name
gdf["output_filename"] = (
    gdf["download_filename"].apply(lambda f: f.replace(".jpg", "_georef.tif"))
)

In [ ]:
gdf

### Calculating extents from center points

In match_multi_zoom, the plugin ultimately calls:
tile_fetcher.fetch_tiles_from_center(
    center_lat,
    center_lon,
    extent_meters,
    extent_dimension="horizontal",
    source_aspect_ratio,
    basemap_source,
    zoom_level
)

The function definition tells us exactly what extent_meters represents:
extent_meters = width (horizontal) OR height (vertical) of the basemap area, in meters [mn365-my.s...epoint.com]
…which is then converted into a rectangle in EPSG:3857:
half_width = width_meters / 2
half_height = height_meters / 2

extent = QgsRectangle(
    center_3857.x() - half_width,
    center_3857.y() - half_height,
    center_3857.x() + half_width,
    center_3857.y() + half_height
)

So the extent_meters must be a real-world width (or height) of your historic air photo footprint.

We just need 1 dimension - probably the height? 

We can get the dimensions and DPI of the photo from the JPEG, but in order to know its real world size, we'll need its scale. 

### How much variation in scale is there due to different flight series? 

https://www.dnr.state.mn.us/maps/historical_airphotos/index.html



In [22]:
# how many different categories are these photos in? 
provider_values = gdf['provider'].nunique()
print(provider_values)

provider_counts = gdf['provider'].value_counts(dropna=False)
print(provider_counts)

print("---------")

project_values = gdf['project'].nunique()
print(project_values)

project_counts = gdf['project'].value_counts(dropna=False)
print(project_counts)

print("---------")
year_values = gdf['year'].nunique()
print(year_values)

year_counts = gdf['year'].value_counts(dropna=False)
print(year_counts)

3
provider
projects    48034
usda        43603
usgs         2124
Name: count, dtype: int64
---------
106
project
stl     9977
slc     8776
czm     8037
cir     7127
koo     4493
        ... 
wn        26
cas       24
eui       24
ca        12
cass       9
Name: count, Length: 106, dtype: int64
---------
44
year
1939    8947
1989    7236
1938    7080
1972    6421
1961    6316
1981    5068
1948    4522
1955    4222
1954    3882
1951    3431
2012    2995
2007    2722
1947    2430
1988    2340
2002    2320
1953    2312
1982    2183
1980    2153
1963    2051
1976    1932
1965    1865
1940    1763
1949    1406
1966    1378
1969    1148
1975    1018
1959     667
1978     442
1957     426
1991     401
1964     361
1973     358
1934     347
1950     339
1960     290
1962     217
1958     143
1941     135
1936     132
1937     130
1952     100
1956      54
1968      54
1979      24
Name: count, dtype: int64


In [ ]:
gdf_sorted = gdf.sort_values(by='provider')

gdf_sorted